# Assignment 4

Deadline: 13.05.2026, 12:00 CET

- Gallo Alessandro , 25-732-140 , alessandro.gallo2@uzh.ch
- Maruccio Anna , 25-742-800 , anna.maruccio@uzh.ch
- Perbellini Cesare, 25-741-257, cesare.perbellini@uzh.ch
- Venturi Matilde , 25-741-059 , matilde.venturi@uzh.ch

In [1]:

# Standard library imports
import os
import sys

# Third party imports
import numpy as np
import pandas as pd
import statsmodels.api as sm    # for regression analysis in 1.c) (or any other regression library you prefer)


project_root = os.path.dirname(os.path.dirname(os.getcwd()))   # Change this path if needed
src_path = os.path.join(project_root, 'qpmwp-course', 'src') #changed for Macbook
sys.path.append(project_root)
sys.path.append(src_path)


# Local modules imports
from helper_functions import (
    load_pickle,
    load_data_spi,
    align_market_data_with_jkp_data,
)
from estimation.covariance import Covariance
from optimization.optimization import PercentilePortfolio
from backtesting.backtest_item_builder.bib_classes import (
    SelectionItemBuilder,
    OptimizationItemBuilder,
)
from backtesting.backtest_item_builder.bibfn_selection import (
    bibfn_selection_gaps,
    bibfn_selection_min_volume,
    bibfn_selection_jkp_data_scores,
)
from backtesting.backtest_item_builder.bibfn_optimization_data import (
    bibfn_scores,
)


from backtesting.backtest_data import BacktestData
from backtesting.backtest_service import BacktestService
from backtesting.backtest import Backtest


## Constants

In [2]:

PATH_TO_DATA =  "../data_backtesting_assignment3/"  # <change this to your path to data>
SAVE_PATH = "../data/"      # <change this to your path where you want to store the backtest>

## Load data and initialize BacktestData class
- market data (from parquet file)
- jkp data (from parquet file)
- swiss performance index, SPI (from csv file)

In [3]:
# Load market and jkp data from parquet files
market_data = pd.read_parquet(path = f'{PATH_TO_DATA}market_data.parquet')
jkp_data = pd.read_parquet(path = f'{PATH_TO_DATA}jkp_data.parquet')
spi = load_data_spi(path='../data/')

# Align market data with jkp data
market_data_ffill, jkp_data = align_market_data_with_jkp_data(market_data=market_data, jkp_data=jkp_data)

# Instantiate the BacktestData class
# and set the market, jkp, and benchmark data as attributes
data = BacktestData()
data.market_data = market_data_ffill  # notice that we use the forward filled market data here
data.jkp_data = jkp_data
data.bm_series = spi

## Define a grid or rebalancing dates

In [4]:
n_month = 3 # We want to rebalance every n_month months
jkp_data_dates = (
    jkp_data
    .index.get_level_values('date')
    .unique().sort_values()
)
rebdates = (
    jkp_data_dates[
        jkp_data_dates > market_data.index.get_level_values('date').min()
    ][::n_month]
    .strftime('%Y-%m-%d').tolist()
)
rebdates = [date for date in rebdates if date > '2002-01-01']
rebdates

['2002-01-31',
 '2002-04-30',
 '2002-07-31',
 '2002-10-31',
 '2003-01-31',
 '2003-04-30',
 '2003-07-31',
 '2003-10-31',
 '2004-01-31',
 '2004-04-30',
 '2004-07-31',
 '2004-10-31',
 '2005-01-31',
 '2005-04-30',
 '2005-07-31',
 '2005-10-31',
 '2006-01-31',
 '2006-04-30',
 '2006-07-31',
 '2006-10-31',
 '2007-01-31',
 '2007-04-30',
 '2007-07-31',
 '2007-10-31',
 '2008-01-31',
 '2008-04-30',
 '2008-07-31',
 '2008-10-31',
 '2009-01-31',
 '2009-04-30',
 '2009-07-31',
 '2009-10-31',
 '2010-01-31',
 '2010-04-30',
 '2010-07-31',
 '2010-10-31',
 '2011-01-31',
 '2011-04-30',
 '2011-07-31',
 '2011-10-31',
 '2012-01-31',
 '2012-04-30',
 '2012-07-31',
 '2012-10-31',
 '2013-01-31',
 '2013-04-30',
 '2013-07-31',
 '2013-10-31',
 '2014-01-31',
 '2014-04-30',
 '2014-07-31',
 '2014-10-31',
 '2015-01-31',
 '2015-04-30',
 '2015-07-31',
 '2015-10-31',
 '2016-01-31',
 '2016-04-30',
 '2016-07-31',
 '2016-10-31',
 '2017-01-31',
 '2017-04-30',
 '2017-07-31',
 '2017-10-31',
 '2018-01-31',
 '2018-04-30',
 '2018-07-

## 1. a) Define the key data fields that characterize a factor theme

- Beside the pre-defined fields, choose three factor themes from table 9 of the Global Factor Data Documentation (See https://jkpfactors-data.s3.amazonaws.com/documents/Documentation.pdf, Section 10.) and add the individual fields.

**(2 points)**

In [5]:
#guardare da pag 38 

JKP_FIELDS_QUALITY = [
    'at_turnover',
    'cop_at',
    'cop_atl1',
    'dgp_dsale',
    'gp_at',
    'gp_atl1',
    'mispricing_perf',
    'ni_inc8q',
    'niq_at',
    'op_at',
    'op_atl1',
    'opex_at',
    'qmj_prof',
    'qmj_growth',
    'qmj_safety',
    'sale_bev',
]

JKP_FIELDS_VALUE = [
    'at_me',
    'be_me',
    'bev_mev',
    'debt_me',
    'div12m_me',
    'ebitda_mev',
    'eq_dur',
    'eqnetis_at',
    'eqnpo_12m',
    'eqnpo_me',
    'fcf_me',
    'ival_me',
    'netis_at',
    'ni_me',
    'ocf_me',
    'sale_me',
]

JKP_FIELDS_MOMENTUM = [
    'prc_highprc_252d',
    'resff3_6_1',
    'resff3_12_1',
    'ret_3_1',
    'ret_6_1',
    'ret_9_1',
    'ret_12_1',
    'seas_1_1na',
]

# <add the name of the factor theme here>
JKP_FIELDS_INVESTMENTS= [
    'aliq_at',
    'at_gr1',
    'be_gr1a',
    'capx_gr1',
    'capx_gr2',
    'capx_gr3',
    'coa_gr1a',
    'col_gr1a',
    'emp_gr1',
    'inv_gr1',
    'inv_gr1a',
    'lnoa_gr1a',
    'mispricing_mgmt',
    'ncoa_gr1a',
    'nncoa_gr1a',
    'noa_gr1a',
    'ppeinv_gr1a',
    'ret_60_12',
    'sale_gr1',
    'sale_gr3',
    'saleq_gr1',
    'seas_2_5na',
]

# <add the name of the factor theme here>
JKP_FIELDS_SIZE = [
    'ami_126d',
    'dolvol_126d',
    'market_equity',
    'prc',
    'rd_me',
]

# <add the name of the factor theme here>
JKP_FIELDS_PROFITABILITY = [
    'dolvol_var_126d',
    'ebit_bev',
    'ebit_sale',
    'f_score',
    'ni_be',
    'niq_be',
    'o_score',
    'ocf_at',
    'ope_be',
    'ope_bel1',
    'turnover_var_126d',
]


JKP_FIELDS = {
    'quality': JKP_FIELDS_QUALITY,
    'value': JKP_FIELDS_VALUE,
    'momentum': JKP_FIELDS_MOMENTUM,
    'inv': JKP_FIELDS_INVESTMENTS,
    'size': JKP_FIELDS_SIZE,
    'profit': JKP_FIELDS_PROFITABILITY,
}

## 1. b) Factor series for each factor theme.

- For each factor theme, run a backtest for the top‑quintile portfolio and a second backtest for the bottom‑quintile portfolio.

- Simulate the return streams for both backtests and define the factor series as a long-short portfolio going long the top quintile portfolio and going short the bottom quintile portfolio.

- Plot the cumulative returns of the top quintile portfolio, the bottom quintile portfolio, the long-short factor portfolio, and the benchmark series.

**(8 points)**

In [ ]:
return_series = data.get_return_series()

factor_results = {}

for factor_theme, factor_fields in JKP_FIELDS.items():

    selection_item_builders = {
        'gaps': SelectionItemBuilder(
            bibfn=bibfn_selection_gaps,
            width=365 * 3,
            n_days=10,
        ),
        'min_volume': SelectionItemBuilder(
            bibfn=bibfn_selection_min_volume,
            width=365,
            min_volume=500_000,
            agg_fn=np.median,
        ),
        'jkp_data_scores': SelectionItemBuilder(
            bibfn=bibfn_selection_jkp_data_scores,
            fields=factor_fields,
        ),
    }


    optimization_item_builders = {
        'scores': OptimizationItemBuilder(
            bibfn=bibfn_scores,
            fields=factor_fields,
        ),
    }

    bs = BacktestService(
        data=data,
        selection_item_builders=selection_item_builders,
        optimization_item_builders=optimization_item_builders,
        rebdates=rebdates,
        quiet=True,
    )

   
    bs.optimization = PercentilePortfolio(
        percentile=80,
        sign='>=',
    )
    bt_top = Backtest()
    bt_top.run(bs=bs)

    
    bs.optimization = PercentilePortfolio(
        percentile=20,
        sign='<=',
    )
    bt_bottom = Backtest()
    bt_bottom.run(bs=bs)

   
    sim_top = bt_top.strategy.simulate(
        return_series=return_series,
        fc=0,
        vc=0,
    ).dropna()

    sim_bottom = bt_bottom.strategy.simulate(
        return_series=return_series,
        fc=0,
        vc=0,
    ).dropna()

    sim_df = pd.concat({
        'Top Quintile': sim_top,
        'Bottom Quintile': sim_bottom,
        'Benchmark': data.bm_series,
    }, axis=1).dropna()


    sim_df['Long-Short Factor'] = (
        sim_df['Bottom Quintile'] - sim_df['Top Quintile']
    )

    factor_results[factor_theme] = sim_df

    np.log(1 + sim_df).cumsum().plot(
        figsize=(12, 7),
        title=f'Cumulative Returns - {factor_theme}'
    )

## 1. c) Factor analysis

- First, compute a factor-mix return series by averaging the returns of the top-quintile portfolio simulations that you have computed above. 

- Second, run an ordinary least squares regression of y on X, where y is your factor-mix series and X contains i) a constant, ii) the SPI return series, and iii) the factor return series computed in 1.b).

- Use monthly returns.

- Print a summary table of the regression output

**(5 points)**

In [ ]:
import pandas as pd
import statsmodels.api as sm


def to_monthly_returns(series):
    s = pd.Series(series).dropna().copy()
    s.index = pd.to_datetime(s.index)
    return (1 + s).resample("ME").prod() - 1

top_quintile_monthly = {}
long_short_monthly = {}

for factor_theme, sim_df in factor_results.items():
    top_quintile_monthly[factor_theme] = to_monthly_returns(sim_df["Top Quintile"])
    long_short_monthly[factor_theme] = to_monthly_returns(sim_df["Long-Short Factor"])

top_quintile_monthly = pd.DataFrame(top_quintile_monthly)
long_short_monthly = pd.DataFrame(long_short_monthly)

spi_monthly = to_monthly_returns(data.bm_series)
spi_monthly.name = "SPI"

factor_mix = top_quintile_monthly.mean(axis=1) #for each month calulate mean top quantile for each factor
factor_mix.name = "factor_mix"


reg_data = pd.concat(
    [factor_mix, spi_monthly, long_short_monthly],
    axis=1
).dropna()

y = reg_data["factor_mix"]
X = reg_data.drop(columns="factor_mix")
X = sm.add_constant(X)


model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             factor_mix   R-squared:                       0.794
Model:                            OLS   Adj. R-squared:                  0.789
Method:                 Least Squares   F-statistic:                     142.8
Date:                Thu, 23 Apr 2026   Prob (F-statistic):           4.03e-85
Time:                        21:19:51   Log-Likelihood:                 620.53
No. Observations:                 267   AIC:                            -1225.
Df Residuals:                     259   BIC:                            -1196.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       2.583e-05      0.002      0.017      0.9